# Ecuaciones diferenciales exactas con SymPy

La meta es dejar que `dsolve` resuelva una EDO con el método de ecuaciones exactas. Ejecuta las celdas en orden con `Shift + Enter`.

## Problema

Resolver, en una región donde $e^x+y\neq 0$,

$$y'=-\frac{e^x y+2x}{e^x+y}, \qquad y(0)=0.$$

In [1]:
import sympy as sp

x, Y = sp.symbols('x y', real=True)
y = sp.Function('y')

P = sp.exp(x)*Y + 2*x
Q = sp.exp(x) + Y
rhs = -P / Q

P_xy = P.subs(Y, y(x))
Q_xy = Q.subs(Y, y(x))
edo_en_y_prima = sp.Eq(sp.diff(y(x), x), rhs.subs(Y, y(x)))
edo = sp.Eq(P_xy + Q_xy*sp.diff(y(x), x), 0)

{'forma en y prima': edo_en_y_prima, 'entrada para dsolve': edo}

{'forma en y prima': Eq(Derivative(y(x), x), (-2*x - y(x)*exp(x))/(y(x) + exp(x))),
 'entrada para dsolve': Eq(2*x + (y(x) + exp(x))*Derivative(y(x), x) + y(x)*exp(x), 0)}

## Solución general con `dsolve`

Indicamos directamente el método `1st_exact`. SymPy devuelve las dos ramas explícitas de la solución general.

In [2]:
solucion_general = sp.dsolve(
    edo,
    y(x),
    hint='1st_exact',
)
solucion_general

[Eq(y(x), -sqrt(C1 - 2*x**2 + exp(2*x)) - exp(x)),
 Eq(y(x), sqrt(C1 - 2*x**2 + exp(2*x)) - exp(x))]

## Solución que cumple $y(0)=0$

Pasamos la condición inicial mediante `ics`. `dsolve` determina la constante y selecciona la rama compatible.

In [3]:
solucion = sp.dsolve(
    edo,
    y(x),
    hint='1st_exact',
    ics={y(0): 0},
)
solucion

Eq(y(x), sqrt(-2*x**2 + exp(2*x)) - exp(x))

## Validación automática

Sustituimos la fórmula obtenida en la forma original $y'=\text{rhs}$ y comprobamos la condición inicial.

In [4]:
formula = solucion.rhs
residuos = {
    'EDO': sp.simplify(
        sp.diff(formula, x) - rhs.subs(Y, formula)
    ),
    'y(0)': sp.simplify(formula.subs(x, 0)),
}

assert all(valor == 0 for valor in residuos.values())
residuos

{'EDO': 0, 'y(0)': 0}

El motor obtiene

$$\boxed{y(x)=-e^x+\sqrt{e^{2x}-2x^2}}.$$

La fórmula se considera en el intervalo alrededor de $x=0$ donde el radicando es no negativo y $e^x+y(x)\neq 0$.